In [ ]:
!pip install -q diffusers transformers accelerate sentencepiece protobuf bitsandbytes
import torch
from diffusers import FluxPipeline, FluxTransformer2DModel, BitsAndBytesConfig

model_id = "sayakpaul/FLUX.1-merged"
quant_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
transformer = FluxTransformer2DModel.from_pretrained(model_id, subfolder="transformer", quantization_config=quant_config, torch_dtype=torch.float16)

pipe = FluxPipeline.from_pretrained(model_id, transformer=transformer, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()

In [ ]:
prompt = "black forest toast spelling out the words 'FLUX'"
width = 1024
height = 1024
steps = 8
guidance = 3.5
seed = 0

generator = torch.Generator("cpu").manual_seed(seed) if seed != 0 else None
result = pipe(prompt, width=width, height=height, guidance_scale=guidance, num_inference_steps=steps, max_sequence_length=256, generator=generator)
image = next(iter(result.images))
image.save("flux.png")
image